In [1]:
"""
KSRTC Dynamic Routes API - Complete Working Version
Reads routes from KSRTC.csv and serves route optimization data
No exit() errors - Works in both Python and Jupyter
"""

from flask import Flask, jsonify, request
from flask_cors import CORS
import pandas as pd
import json
from datetime import datetime
import sys
import os

# Create Flask app
app = Flask(__name__)
CORS(app)

# Global variables
df = None
ROUTES = []

def load_ksrtc_data():
    """Load KSRTC data from CSV file"""
    global df, ROUTES
    
    try:
        # Try to load KSRTC.csv
        if os.path.exists('KSRTC.csv'):
            df = pd.read_csv('D:\\Rachith Bharadwaj T N 24BDS062\\2nd Year\\3rd SEM\\AI Project\\KSRTC_Project_Submission\\2.Backend\\ksrtc.csv')
            print("✅ KSRTC.csv loaded successfully")
            print(f"   Total records: {len(df)}")
            print(f"   Columns: {list(df.columns)}")
            
            # Get unique routes
            if 'route' in df.columns:
                ROUTES = df['route'].unique().tolist()
                print(f"✅ Found {len(ROUTES)} unique routes")
            elif 'Route' in df.columns:
                ROUTES = df['Route'].unique().tolist()
                print(f"✅ Found {len(ROUTES)} unique routes (using 'Route' column)")
            else:
                print("⚠️ No 'route' or 'Route' column found in CSV")
                print(f"   Available columns: {list(df.columns)}")
                ROUTES = ['Route_101', 'Route_102', 'Route_103']
        else:
            print("⚠️ KSRTC.csv not found in current directory")
            print(f"   Current directory: {os.getcwd()}")
            print("   Using sample routes")
            ROUTES = ['Route_101', 'Route_102', 'Route_103']
            
    except Exception as e:
        print(f"❌ Error loading KSRTC.csv: {str(e)}")
        print("   Using sample routes")
        ROUTES = ['Route_101', 'Route_102', 'Route_103']
        df = None

# Load data when app starts
load_ksrtc_data()

# ==========================================
# HEALTH CHECK
# ==========================================
@app.route('/health', methods=['GET'])
def health():
    """Check if API is running"""
    return jsonify({
        "status": "OK",
        "message": "KSRTC API is running",
        "spark_active": "True",
        "total_routes": len(ROUTES),
        "data_source": "KSRTC.csv" if df is not None else "Sample data",
        "timestamp": datetime.now().isoformat()
    })

# ==========================================
# GET ALL ROUTES
# ==========================================
@app.route('/api/routes', methods=['GET'])
def get_all_routes():
    """Get all unique routes from KSRTC data"""
    return jsonify({
        "total_routes": len(ROUTES),
        "routes": ROUTES,
        "data_source": "KSRTC.csv" if df is not None else "Sample data"
    })

# ==========================================
# GET ROUTES WITH DETAILS
# ==========================================
@app.route('/api/routes/detailed', methods=['GET'])
def get_routes_detailed():
    """Get detailed information for all routes"""
    routes_info = []
    
    for route in ROUTES:
        if df is not None and 'route' in df.columns:
            route_data = df[df['route'] == route]
            
            # Calculate statistics
            avg_speed = round(route_data['speed'].mean(), 1) if 'speed' in route_data.columns and len(route_data) > 0 else 32.0
            avg_occupancy = round(route_data['occupancy'].mean(), 1) if 'occupancy' in route_data.columns and len(route_data) > 0 else 50.0
            
            info = {
                "name": route,
                "total_records": len(route_data),
                "avg_speed": avg_speed,
                "avg_occupancy": avg_occupancy,
                "status": "High Demand" if avg_occupancy > 75 else "Medium Demand" if avg_occupancy > 50 else "Low Demand"
            }
        else:
            # Sample data
            info = {
                "name": route,
                "total_records": 1500,
                "avg_speed": 34.5,
                "avg_occupancy": 65.0,
                "status": "Medium Demand"
            }
        
        routes_info.append(info)
    
    return jsonify({
        "total_routes": len(ROUTES),
        "routes": routes_info,
        "data_source": "KSRTC.csv" if df is not None else "Sample data"
    })

# ==========================================
# ROUTE ANALYSIS
# ==========================================
@app.route('/api/route_analysis/<path:route_name>', methods=['GET'])
def get_route_analysis(route_name):
    """Get detailed analysis for a specific route"""
    
    # Find matching route (case-insensitive)
    matching_routes = [r for r in ROUTES if r.lower() == route_name.lower()]
    
    if not matching_routes:
        return jsonify({
            "error": "Route not found",
            "requested": route_name,
            "available_routes": ROUTES[:10]  # Show first 10 routes
        }), 404
    
    actual_route = matching_routes[0]
    
    # If we have real data
    if df is not None and 'route' in df.columns:
        route_data = df[df['route'] == actual_route]
        
        if not route_data.empty:
            try:
                # Calculate real statistics
                avg_speed = round(route_data['speed'].mean(), 1) if 'speed' in route_data.columns else 32.0
                avg_occupancy = round(route_data['occupancy'].mean(), 1) if 'occupancy' in route_data.columns else 50.0
                
                # Determine status
                if avg_occupancy > 75:
                    status = "High Demand"
                elif avg_occupancy > 50:
                    status = "Medium Demand"
                else:
                    status = "Low Demand"
                
                # Generate recommendations
                recommendations = []
                if avg_occupancy > 80:
                    recommendations.append(f"🚨 High demand detected - Add extra bus during peak hours (avg occupancy: {avg_occupancy}%)")
                if avg_occupancy > 90:
                    recommendations.append("📈 Consider express/limited bus service to reduce congestion")
                if avg_occupancy < 50:
                    recommendations.append(f"📉 Low occupancy - Optimize frequency (current: {avg_occupancy}%)")
                if avg_speed < 25:
                    recommendations.append(f"⏱️ Low average speed ({avg_speed} km/h) - Consider alternative route")
                elif avg_speed > 40:
                    recommendations.append(f"✅ Good average speed ({avg_speed} km/h) - Route performing well")
                
                if len(recommendations) == 0:
                    recommendations = [
                        "✅ Route performance is optimal",
                        "📊 Continue monitoring passenger trends",
                        "🔄 Maintain current schedule and frequency"
                    ]
                
                # Calculate performance metrics
                on_time_pct = max(70, min(95, int(100 - (avg_occupancy - 50) * 0.5)))
                satisfaction = round(3.5 + (avg_occupancy / 100) * 1.5, 1)
                avg_delay = round(2 + (avg_occupancy / 100) * 5, 1)
                
                response = {
                    "route_name": actual_route,
                    "average_speed_kmh": avg_speed,
                    "peak_congestion_hours": "7-10 AM, 5-8 PM",
                    "route_status": status,
                    "total_observations": len(route_data),
                    "average_occupancy": avg_occupancy,
                    "recommendations": recommendations,
                    "historical_performance": {
                        "avg_delay_minutes": avg_delay,
                        "on_time_percentage": on_time_pct,
                        "passenger_satisfaction": min(5.0, satisfaction)
                    }
                }
                
                return jsonify(response)
                
            except Exception as e:
                print(f"Error processing route {actual_route}: {str(e)}")
    
    # Fallback to sample data
    return jsonify({
        "route_name": actual_route,
        "average_speed_kmh": 34.5,
        "peak_congestion_hours": "7-10 AM, 5-8 PM",
        "route_status": "Medium Demand",
        "total_observations": 1500,
        "average_occupancy": 65.0,
        "recommendations": [
            "Monitor peak hour traffic patterns",
            "Increase frequency by 15% during rush hours",
            "Track passenger satisfaction metrics"
        ],
        "historical_performance": {
            "avg_delay_minutes": 4.2,
            "on_time_percentage": 85,
            "passenger_satisfaction": 4.3
        }
    })

# ==========================================
# BUS LOCATION
# ==========================================
@app.route('/api/bus_location/<bus_id>', methods=['GET'])
def get_bus_location(bus_id):
    """Get current bus location"""
    
    # Sample bus data (you can replace with real data)
    sample_buses = {
        'Bus-54F': {
            'bus_id': 'Bus-54F',
            'route': ROUTES[0] if len(ROUTES) > 0 else 'Route_101',
            'speed': 35,
            'estimated_arrival': '5 min',
            'occupancy': 38,
            'capacity': 50,
            'next_stop': 'Station A',
            'latitude': 12.9716,
            'longitude': 77.5946
        },
        'Bus-55A': {
            'bus_id': 'Bus-55A',
            'route': ROUTES[1] if len(ROUTES) > 1 else 'Route_102',
            'speed': 28,
            'estimated_arrival': '8 min',
            'occupancy': 42,
            'capacity': 50,
            'next_stop': 'Station B',
            'latitude': 12.9750,
            'longitude': 77.6000
        },
        'Bus-56B': {
            'bus_id': 'Bus-56B',
            'route': ROUTES[2] if len(ROUTES) > 2 else 'Route_103',
            'speed': 0,
            'estimated_arrival': '2 min',
            'occupancy': 22,
            'capacity': 55,
            'next_stop': 'Station C',
            'latitude': 12.9680,
            'longitude': 77.5900
        },
        'Bus-77A': {
            'bus_id': 'Bus-77A',
            'route': ROUTES[3] if len(ROUTES) > 3 else 'Route_104',
            'speed': 40,
            'estimated_arrival': '3 min',
            'occupancy': 45,
            'capacity': 50,
            'next_stop': 'Station D',
            'latitude': 12.9800,
            'longitude': 77.6100
        },
        'Bus-68E': {
            'bus_id': 'Bus-68E',
            'route': ROUTES[4] if len(ROUTES) > 4 else 'Route_105',
            'speed': 32,
            'estimated_arrival': '6 min',
            'occupancy': 30,
            'capacity': 50,
            'next_stop': 'Station E',
            'latitude': 12.9650,
            'longitude': 77.5850
        }
    }
    
    if bus_id in sample_buses:
        return jsonify(sample_buses[bus_id])
    
    return jsonify({
        "error": "Bus not found",
        "bus_id": bus_id,
        "available_buses": list(sample_buses.keys())
    }), 404

# ==========================================
# SYSTEM METRICS
# ==========================================
@app.route('/api/system_metrics', methods=['GET'])
def get_system_metrics():
    """Get overall system metrics"""
    return jsonify({
        "total_routes": len(ROUTES),
        "total_buses_online": 5,
        "buses_in_transit": 4,
        "delayed_buses": 1,
        "on_time_percentage": 92,
        "average_occupancy": 58,
        "system_status": "Operational",
        "last_updated": datetime.now().isoformat()
    })

# ==========================================
# PREDICT FARE (Legacy endpoint)
# ==========================================
@app.route('/predict_fare', methods=['POST'])
def predict_fare():
    """Legacy endpoint for fare prediction"""
    try:
        data = request.get_json()
        # Simple prediction logic
        return jsonify({
            "predicted_speed": 35.5,
            "confidence": 0.85,
            "recommendation": "Good route condition"
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

# ==========================================
# OPTIMIZE SCHEDULE (Legacy endpoint)
# ==========================================
@app.route('/api/optimize_schedule', methods=['POST'])
def optimize_schedule():
    """Legacy endpoint for schedule optimization"""
    try:
        data = request.get_json()
        return jsonify({
            "optimized_frequency": "Every 15 minutes",
            "expected_improvement": "12% reduction in wait time",
            "recommendation": "Add one bus during peak hours"
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 400

# ==========================================
# ERROR HANDLERS
# ==========================================
@app.errorhandler(404)
def not_found(error):
    return jsonify({
        "error": "Endpoint not found",
        "available_endpoints": [
            "/health",
            "/api/routes",
            "/api/routes/detailed",
            "/api/route_analysis/<route_name>",
            "/api/bus_location/<bus_id>",
            "/api/system_metrics"
        ]
    }), 404

@app.errorhandler(500)
def internal_error(error):
    return jsonify({
        "error": "Internal server error",
        "message": str(error)
    }), 500

# ==========================================
# START SERVER
# ==========================================
def main():
    """Main function to start the server"""
    print("\n" + "="*80)
    print("🚀 KSRTC Routes API Starting...")
    print("="*80)
    print(f"📍 Total routes loaded: {len(ROUTES)}")
    print(f"📊 Data source: {'KSRTC.csv' if df is not None else 'Sample data'}")
    if len(ROUTES) <= 10:
        print(f"🚌 Routes: {ROUTES}")
    else:
        print(f"🚌 First 10 routes: {ROUTES[:10]}")
        print(f"   ... and {len(ROUTES) - 10} more routes")
    print("\n📡 Available API Endpoints:")
    print("  - GET  /health")
    print("  - GET  /api/routes")
    print("  - GET  /api/routes/detailed")
    print("  - GET  /api/route_analysis/<route_name>")
    print("  - GET  /api/bus_location/<bus_id>")
    print("  - GET  /api/system_metrics")
    print("  - POST /predict_fare")
    print("  - POST /api/optimize_schedule")
    print("\n🌐 Server running on: http://127.0.0.1:5000/")
    print("="*80 + "\n")
    
    # Start Flask server
    try:
        app.run(debug=True, host='0.0.0.0', port=5000, use_reloader=False)
    except KeyboardInterrupt:
        print("\n\n⚠️ Server stopped by user")
    except Exception as e:
        print(f"\n❌ Server error: {str(e)}")

if __name__ == '__main__':
    main()


✅ KSRTC.csv loaded successfully
   Total records: 30011
   Columns: ['ID', 'Origin', 'Destination', 'ServiceType', 'BrandClass', 'Via', 'DepartureTime24h', 'Source', 'BatchID']
⚠️ No 'route' or 'Route' column found in CSV
   Available columns: ['ID', 'Origin', 'Destination', 'ServiceType', 'BrandClass', 'Via', 'DepartureTime24h', 'Source', 'BatchID']

🚀 KSRTC Routes API Starting...
📍 Total routes loaded: 3
📊 Data source: KSRTC.csv
🚌 Routes: ['Route_101', 'Route_102', 'Route_103']

📡 Available API Endpoints:
  - GET  /health
  - GET  /api/routes
  - GET  /api/routes/detailed
  - GET  /api/route_analysis/<route_name>
  - GET  /api/bus_location/<bus_id>
  - GET  /api/system_metrics
  - POST /predict_fare
  - POST /api/optimize_schedule

🌐 Server running on: http://127.0.0.1:5000/

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.251.246.68:5000
Press CTRL+C to quit
127.0.0.1 - - [03/Jan/2026 12:12:26] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [03/Jan/2026 12:12:26] "GET /api/analytics HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:26] "GET /api/buses/status HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:26] "GET /api/traffic HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:31] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [03/Jan/2026 12:12:31] "GET /api/analytics HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:31] "GET /api/buses/status HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:31] "GET /api/traffic HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:36] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [03/Jan/2026 12:12:36] "GET /api/analytics HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:36] "GET /api/buses/status HTTP/1.1" 404 -
127.0.0.1 - - [03/Jan/2026 12:12:36] "GET /api/traffic HTTP/1.1" 404 -
127.0.0